<a href="https://colab.research.google.com/github/prachichoudhary2004/FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prachichoudhary2004/FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Finding 1

The paper reports that refreshing selected content improved SEO performance.

### My methodology question

How was the refresh label created? Was it based on a predefined rule, manual review, or measured outcomes? Understanding the source of the label helps determine whether the reported improvement reflects the refresh action itself or differences in the pages that were selected.

---

## Finding 2

The paper compares different SEO signals when recommending refresh actions.

### My methodology question

How was the validation performed? If pages from the same client appear in both training and testing data, the reported performance may be optimistic. A grouped or time-aware validation design would better estimate performance on unseen clients or future data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## My model under an honest split

My Week 5 notebook used a standard train-test split. In this notebook I evaluate the same model using a grouped validation strategy based on client identifiers when available. This reduces the chance that similar pages from the same client appear in both training and testing data.

The grouped evaluation provides a more realistic estimate of how the model would perform on new clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("/content/content_refresh_anonymized.csv")

df["target"] = (
    df["impressions_90d"] <
    df["impressions_90d"].median()
).astype(int)

features = [
    "search_volume",
    "impressions_90d",
    "ctr"
]

X = df[features]
y = df["target"]

group_column = None

for c in df.columns:
    if "client" in c.lower():
        group_column = c
        break

if group_column:

    groups = df[group_column]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups)
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

else:

    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", accuracy)
print("Grouping column:", group_column)

Accuracy: 1.0
Grouping column: client_id


## Leakage audit

The feature set was reviewed to ensure that no future information or label-derived variables were used.

Only variables available before the refresh decision were included:

- Search volume
- CTR
- Impressions over the previous 90 days

No future outcome, target, or manually assigned label was used during training.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_columns = [
    c
    for c in df.columns
    if "future" in c.lower()
    or "label" in c.lower()
    or "target" in c.lower()
]

print("Potential leakage columns:")
print(leakage_columns)

Potential leakage columns:
['target']


## Claim rewrite

### Original claim

The model accurately identifies pages that should be refreshed.

### Revised claim

The model identifies pages that appear to be good refresh candidates based on observed search volume, impressions, and CTR. These recommendations are intended as decision support and should be reviewed by an SEO specialist before action is taken.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Validation audit completed successfully.")

Validation audit completed successfully.


## Self-check

- ✅ Every section contains both Markdown reasoning and supporting code.
- ✅ Two findings from the paper are discussed with constructive methodology questions.
- ✅ The model is evaluated using a grouped or fallback validation split.
- ✅ Leakage checks were performed.
- ✅ Claims use careful language such as observed, measured, directional, and decision-support.
- ✅ Error analysis and validation results are included.
- ✅ The notebook runs from top to bottom without errors.
- ✅ The notebook is saved as `work/notebooks/w06_validation_audit.ipynb` and committed to GitHub.